<a href="https://colab.research.google.com/github/2121045-eng/dongsa/blob/main/streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install streamlit
import streamlit as st
import json
import os

# --- 설정 및 데이터 로드 (기존 로직 유지) ---
KNOWLEDGE_BASE_FILE = 'knowledge_base.json'

def load_knowledge_base():
    default_knowledge_base_data = {
        "호랑이": ["줄무늬가 있어요.", "산의 왕이라고 불려요.", "'어흥' 하고 무서운 소리를 내요."],
        "사과": ["빨간색 과일이에요.", "맛이 달콤하고 아삭아삭해요.", "백설공주가 먹었던 과일이에요."],
        "기차": ["아주 길어요.", "철길 위로 다녀요.", "'칙칙폭폭' 소리가 나요."],
        "바나나": ["길쭉하고 노란색이에요.", "껍질을 까서 먹어야 해요.", "원숭이가 아주 좋아해요."],
        "우산": ["비가 올 때 사용해요.", "머리 위로 써서 옷이 젖지 않게 해줘요.", "다 쓰면 접어서 보관해요."],
        "망치": ["못을 박을 때 사용하는 공구예요.", "머리 부분이 무거운 쇠로 되어 있어요.", "손이 다치지 않게 조심해서 써야 해요."],
        "우체통": ["편지를 넣으면 집으로 배달해 줘요.", "길거리에서 볼 수 있는 빨간색 통이에요.", "요즘은 찾아보기 조금 어려워졌어요."],
        "빗자루": ["바닥의 먼지나 쓰레기를 쓸어내요.", "길쭉한 막대 끝에 솔이 달려 있어요.", "쓰레받기와 짝꿍이에요."],
        "반창고": ["상처가 난 곳에 붙이는 스티커예요.", "피가 나거나 긁혔을 때 보호해 줘요.", "가운데에 부드러운 솜이 붙어 있어요."],
        "칫솔": ["치약을 묻혀서 이를 깨끗하게 닦아요.", "입안의 세균을 없애주는 도구예요.", "밥을 먹고 난 뒤에 꼭 사용해요."],
    }

    if os.path.exists(KNOWLEDGE_BASE_FILE):
        with open(KNOWLEDGE_BASE_FILE, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
                return data if isinstance(data, dict) else default_knowledge_base_data
            except:
                return default_knowledge_base_data
    return default_knowledge_base_data

def save_knowledge_base(data):
    with open(KNOWLEDGE_BASE_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

# 지식 베이스 초기화
if 'kb' not in st.session_state:
    st.session_state.kb = load_knowledge_base()

# --- UI 레이아웃 시작 ---
st.set_page_config(page_title="단어 마스터의 비밀 퀴즈", page_icon="🕵️‍♂️")
st.title("# 🕵️‍♂️ 단어 마스터의 비밀 퀴즈")

# Streamlit의 Tabs 사용
tab1, tab2, tab3 = st.tabs(["🔍 수수께끼 생성하기", "✨ 새로운 단어 추가", "✏️ 단어 수정하기"])

# --- Tab 1: 수수께끼 생성하기 ---
with tab1:
    st.markdown("정답 단어를 입력하면, AI가 아동의 추론 훈련을 위한 3가지 힌트를 보여줍니다.")
    # [widgets.Textarea -> st.text_input / text_area]
    word_input = st.text_input("수수께끼 정답 단어", placeholder="예: 호랑이, 사과, 기차...")

    # [widgets.Button -> if st.button:]
    if st.button("수수께끼 생성하기 🔍"):
        word = word_input.strip()
        if not word:
            st.warning("단어를 입력해주세요!")
        elif word in st.session_state.kb:
            hints = st.session_state.kb[word]
            st.markdown(f"""
            ### 🕵️‍♂️ 누구일까요? (수수께끼 힌트)
            1. **{hints[0]}**
            2. **{hints[1]}**
            3. **{hints[2]}**
            ---
            **💡 치료 팁:** 아이에게 바로 정답을 알려주지 말고, 각 힌트를 들을 때마다 머릿속에 그림을 그려보게 하세요.
            아이의 추론 능력을 자극하는 **비계 설정(Scaffolding)**이 중요합니다!
            """)
        else:
            st.error(f"'{word}' 단어에 대한 힌트가 없습니다. '새로운 단어 추가하기' 탭에서 추가해주세요.")

# --- Tab 2: 새로운 단어 추가하기 ---
with tab2:
    st.markdown("지식 베이스에 새로운 단어와 3가지 힌트를 추가합니다.")
    new_word = st.text_input("새로운 단어", key="new_word")
    h1 = st.text_input("힌트 1", key="h1")
    h2 = st.text_input("힌트 2", key="h2")
    h3 = st.text_input("힌트 3", key="h3")

    if st.button("단어 추가하기 ✨"):
        if new_word and h1 and h2 and h3:
            st.session_state.kb[new_word.strip()] = [h1.strip(), h2.strip(), h3.strip()]
            save_knowledge_base(st.session_state.kb)
            st.success(f"✅ '{new_word}' 단어가 성공적으로 추가되었습니다!")
        else:
            st.error("모든 필드를 채워주세요!")

# --- Tab 3: 단어 수정하기 ---
with tab3:
    st.markdown("이미 등록된 단어의 힌트를 수정합니다.")
    # 드롭다운에서 선택
    options = list(st.session_state.kb.keys())
    selected_word = st.selectbox("수정할 단어 선택", options)

    if selected_word:
        current_hints = st.session_state.kb[selected_word]
        m1 = st.text_input("새 힌트 1", value=current_hints[0])
        m2 = st.text_input("새 힌트 2", value=current_hints[1])
        m3 = st.text_input("새 힌트 3", value=current_hints[2])

        if st.button("단어 힌트 수정하기 ✏️"):
            st.session_state.kb[selected_word] = [m1.strip(), m2.strip(), m3.strip()]
            save_knowledge_base(st.session_state.kb)
            st.success(f"✏️ '{selected_word}' 단어가 수정되었습니다!")
            # 페이지를 새로고침하여 다른 탭에도 반영되게 함
            st.rerun()

2026-04-25 18:21:48.191 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 18:21:48.192 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 18:21:48.194 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 18:21:48.195 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 18:21:48.196 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 18:21:48.197 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 18:21:48.198 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-25 18:21:48.199 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar